In [1]:
!pip install textstat nltk openpyxl
!pip install NRCLex==3.0.0

import json
import pandas as pd
import textstat
import nltk

nltk.download('punkt_tab')
nltk.download('vader_lexicon')

from nltk.tokenize import sent_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
from collections import Counter
from nrclex import NRCLex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 17.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for NRCLex: filename=NRCLex-3.0.0-py3-none-any.whl size=43309 sha256=e3cab3397612a4bafe644c69c9d245558841ec9a27c220c5001f294e54fe2a07
  Stored in directory: /root/.cache/pip/wheels/1f/e8/d0/e3c3da0ef3b37ef4381dbf5c9401f3a9861a63ce221b13d8bb
Successfully built NRCLex


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


Enter 1 for human sample analysis
Enter 2 for machine generated sample analysis
Enter choice: 1


Saving reddit_bloom_7b.raw_data.json to reddit_bloom_7b.raw_data.json
Saving reddit_claude3haiku.raw_data.json to reddit_claude3haiku.raw_data.json
Saving reddit_falcon_7b.raw_data.json to reddit_falcon_7b.raw_data.json
Saving reddit_gemma_7b.raw_data.json to reddit_gemma_7b.raw_data.json
Saving reddit_gpt2_xl.raw_data.json to reddit_gpt2_xl.raw_data.json
Saving reddit_gpt4o.raw_data.json to reddit_gpt4o.raw_data.json
Saving reddit_gpt4turbo.raw_data.json to reddit_gpt4turbo.raw_data.json
Saving reddit_gptj_6b.raw_data.json to reddit_gptj_6b.raw_data.json
Saving reddit_gptneo_2.7b.raw_data.json to reddit_gptneo_2.7b.raw_data.json
Saving reddit_llama1_13b.raw_data.json to reddit_llama1_13b.raw_data.json
Saving reddit_llama2_13b.raw_data.json to reddit_llama2_13b.raw_data.json
Saving reddit_llama3_8b.raw_data.json to reddit_llama3_8b.raw_data.json
Saving reddit_opt_2.7b.raw_data.json to reddit_opt_2.7b.raw_data.json
Saving reddit_opt_13b.raw_data.json to reddit_opt_13b.raw_data.json
Savi

In [ ]:
print("Enter ""1"" for human sample analysis")
print("Enter ""2"" for machine generated sample analysis")
choice=int(input("Enter choice: "))
from google.colab import files
uploaded = files.upload()

In [5]:
def extract_dataset():
  if choice==1:
    originals = []
    for filename in uploaded.keys():
      with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
      originals.extend(data["original"])
    df = pd.DataFrame({"text": originals})
    return df, "Human"
  elif choice==2:
    filename = list(uploaded.keys())[0]
    with open(filename, "r", encoding="utf-8") as f: data = json.load(f)
    df = pd.DataFrame({"text": data["sampled"]})
    return df, filename.replace(".json","")
  else:
    print("Invalid choice")
    return None

def char_count(txt_column): return len(str(txt_column))

def word_count(txt_column): return len(str(txt_column).split())

def sentence_count(txt_column): return len(sent_tokenize(str(txt_column)))

def paragraph_count(txt_column):
  paragraphs = str(txt_column).split('\n\n')
  paragraphs = [ p for p in paragraphs if p.strip() ]
  return len(paragraphs)

def average_word_length(txt_column):
  words = str(txt_column).split()
  if len(words) == 0:
    return 0
  total_chars = sum(len(word) for word in words)
  return total_chars / len(words)

def average_sentence_length(txt_column):
  sentences = sentence_count(txt_column)
  if sentences == 0:
    return 0
  return word_count(txt_column) / sentences

def lexical_diversity(txt_column):
  words = str(txt_column).lower().split()
  if len(words) == 0:
    return 0
  return len(set(words)) / len(words)

def hapax_legomena(txt_column):
  words = str(txt_column).lower().split()
  counts = Counter(words)
  return sum(1 for freq in counts.values() if freq == 1 )

def hapax_ratio(txt_column):
  words = str(txt_column).lower().split()
  if len(words) == 0:
    return 0
  counts = Counter(words)
  hapax = sum( 1 for freq in counts.values() if freq == 1 )
  return hapax / len(words)

def sentiment_positive(txt_column):
    return analyzer.polarity_scores(str(txt_column))["pos"]

def sentiment_negative(txt_column):
    return analyzer.polarity_scores(str(txt_column))["neg"]

def sentiment_neutral(txt_column):
    return analyzer.polarity_scores(str(txt_column))["neu"]

def sentiment_compound(txt_column):
    return analyzer.polarity_scores(str(txt_column))["compound"]

def get_emotion_score(txt_column, emotion):
    try:
        emotions = NRCLex(str(txt_column)).affect_frequencies
        return emotions.get(emotion, 0)
    except:
        return 0

df, name = extract_dataset()
txt_column="text"
summary = {
    "Word Count": df[txt_column].apply(word_count).mean(),
    "Character Count": df[txt_column].apply(char_count).mean(),
    "Sentence Count": df[txt_column].apply(sentence_count).mean(),
    "Paragraph Count": df[txt_column].apply(paragraph_count).mean(),
    "Average Word Length": df[txt_column].apply(average_word_length).mean(),
    "Average Sentence Length": df[txt_column].apply(average_sentence_length).mean(),
    "Lexical Diversity": df[txt_column].apply(lexical_diversity).mean(),
    "Hapax Legomena": df[txt_column].apply(hapax_legomena).mean(),
    "Hapax Ratio": df[txt_column].apply(hapax_ratio).mean(),
    "Flesch Reading Ease": df[txt_column].apply(textstat.flesch_reading_ease).mean(),
    "Flesch-Kincaid Grade": df[txt_column].apply(textstat.flesch_kincaid_grade).mean(),
    "Gunning Fog": df[txt_column].apply(textstat.gunning_fog).mean(),
    "SMOG Index": df[txt_column].apply(textstat.smog_index).mean(),
    "Sentiment Positive": df["text"].apply(sentiment_positive).mean(),
    "Sentiment Negative": df["text"].apply(sentiment_negative).mean(),
    "Sentiment Neutral": df["text"].apply(sentiment_neutral).mean(),
    "Sentiment Compound": df["text"].apply(sentiment_compound).mean(),
    "NRCLex Anger": df["text"].apply(lambda x: get_emotion_score(x, "anger")).mean(),
    "NRCLex Fear": df["text"].apply(lambda x: get_emotion_score(x, "fear")).mean(),
    "NRCLex Anticipation": df["text"].apply(lambda x: get_emotion_score(x, "anticipation")).mean(),
    "NRCLex Trust": df["text"].apply(lambda x: get_emotion_score(x, "trust")).mean(),
    "NRCLex Surprise": df["text"].apply(lambda x: get_emotion_score(x, "surprise")).mean(),
    "NRCLex Sadness": df["text"].apply(lambda x: get_emotion_score(x, "sadness")).mean(),
    "NRCLex Joy": df["text"].apply(lambda x: get_emotion_score(x, "joy")).mean(),
    "NRCLex Disgust": df["text"].apply(lambda x: get_emotion_score(x, "disgust")).mean(),
    "NRCLex Positive": df["text"].apply(lambda x: get_emotion_score(x, "positive")).mean(),
    "NRCLex Negative": df["text"].apply(lambda x: get_emotion_score(x, "negative")).mean()
}

result=pd.DataFrame(list(summary.items()), columns=["Linguistic Features(avg)", "Values"])
output_file = f"{name}_OutputTask1.xlsx"
result.to_excel(output_file,index=False)
print(result)
print(f"Saved successfully as {output_file}")

files.download(output_file)

   Linguistic Features(avg)      Values
0                Word Count  162.282667
1           Character Count  928.404889
2            Sentence Count    9.760889
3           Paragraph Count    1.000000
4       Average Word Length    4.726246
5   Average Sentence Length   18.107470
6         Lexical Diversity    0.679871
7            Hapax Legomena   84.441778
8               Hapax Ratio    0.524754
9       Flesch Reading Ease   61.578296
10     Flesch-Kincaid Grade    9.234491
11              Gunning Fog   11.578373
12               SMOG Index   11.247901
13       Sentiment Positive    0.077617
14       Sentiment Negative    0.065500
15        Sentiment Neutral    0.856876
16       Sentiment Compound    0.144328
17             NRCLex Anger    0.048238
18              NRCLex Fear    0.089521
19      NRCLex Anticipation    0.085244
20             NRCLex Trust    0.123155
21          NRCLex Surprise    0.038152
22           NRCLex Sadness    0.084386
23               NRCLex Joy    0.065493


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>